# Entrenar "oye matix" (openWakeWord, espanol) -- version rapida

Genera el modelo de wake word **"oye matix"** y exporta `oye_matix.onnx` compatible con Matix (cadena mel -> embedding -> clasificador; entrada `[1,16,96]`, salida `[1,1]`).

**IMPORTANTE antes de correr:**
- Entorno de ejecucion -> Cambiar tipo de entorno -> **GPU (T4)**.
- **Manten la laptop ENCHUFADA y que NO se suspenda** y la pestana de Colab abierta hasta el final. Si se apaga/suspende, Colab se desconecta y se pierde todo.
- El aviso 'estas en GPU pero no la usas' es NORMAL (las primeras celdas no usan GPU). NO cambies a CPU.

Tarda ~30-40 min. El modelo final se **guarda en tu Google Drive** para que no se pierda.

In [ ]:
!nvidia-smi

## 1. Instalar openWakeWord + Piper + extras (~3 min)

In [ ]:
!git clone https://github.com/dscripka/openWakeWord
%pip install -q -e ./openWakeWord
%pip install -q piper-tts
%pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
  acoustics==0.2.6 'datasets[audio]==2.14.6'
%pip install -q pronouncing onnx  # pronouncing lo pide openwakeword.data; onnx para la celda 8

### Arreglo de compatibilidad (torchaudio / torch-audiomentations)

In [ ]:
# Compatibilidad: torch-audiomentations llama torchaudio.set_audio_backend()
# (eliminado en torchaudio nuevo) AL IMPORTARSE. Editamos el archivo SIN
# importar el paquete (importarlo es justo lo que revienta).
import glob, re
fs = glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py')
for f in fs:
    s = open(f).read()
    open(f, 'w').write(re.sub(r'torchaudio\.set_audio_backend\([^)]*\)', 'pass', s))
    print('parcheado:', f, '| refs restantes:', open(f).read().count('set_audio_backend'))

## 2. Descargar 8 voces de Piper en espanol

In [ ]:
import os, urllib.request
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/es'
voices = [('es_AR','daniela','high'), ('es_ES','carlfm','x_low'), ('es_ES','davefx','medium'),
          ('es_ES','sharvard','medium'), ('es_ES','mls_9972','low'), ('es_ES','mls_10246','low'),
          ('es_MX','ald','medium'), ('es_MX','claude','high')]
os.makedirs('voices', exist_ok=True)
for loc, name, q in voices:
    stem = f'{loc}-{name}-{q}'
    for ext in ('.onnx', '.onnx.json'):
        try:
            urllib.request.urlretrieve(f'{BASE}/{loc}/{name}/{q}/{stem}{ext}', f'voices/{stem}{ext}')
            print('ok', stem + ext)
        except Exception as e:
            print('FALLO', stem + ext, e)

## 3. Sintetizar "oye matix" (rapido: ~320 train + ~64 test) y remuestrear a 16 kHz

Va imprimiendo el avance cada 50 clips. Con 8 voces + aumentacion fuerte alcanza para un primer modelo. Si quisieras mas, sube `reps` (pero tarda mas).

In [ ]:
import os, subprocess, uuid, itertools, glob
ROOT = '/content/my_custom_model/oye_matix'
os.makedirs(ROOT + '/positive_train', exist_ok=True)
os.makedirs(ROOT + '/positive_test', exist_ok=True)
PHRASE = 'oye matix'
length_scales = [0.9, 1.1]; noise = [0.6]
def synth(outdir, reps):
    vs = [v for v in os.listdir('voices') if v.endswith('.onnx')]
    hechos = 0
    for v in vs:
        for ls, ns in itertools.product(length_scales, noise):
            for _ in range(reps):
                tmp = '/content/_t.wav'; fn = os.path.join(outdir, uuid.uuid4().hex + '.wav')
                try:
                    subprocess.run(['piper', '-m', f'voices/{v}', '-f', tmp,
                                    '--length_scale', str(ls), '--noise_scale', str(ns)],
                                   input=PHRASE.encode(), check=True,
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', fn],
                                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    hechos += 1
                    if hechos % 50 == 0: print('  ...', hechos, 'clips')
                except Exception as e:
                    print('synth FALLO en', v, '->', e); return
    print(outdir, '->', hechos, 'clips')
synth(ROOT + '/positive_train', reps=20)  # ~320
synth(ROOT + '/positive_test',  reps=4)   # ~64
print('TOTAL train:', len(glob.glob(ROOT + '/positive_train/*.wav')),
      '| test:', len(glob.glob(ROOT + '/positive_test/*.wav')))

## 4. Descargar negativos + RIR + features precomputadas

In [ ]:
import os
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('mit_rirs', exist_ok=True)
snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset', local_dir='mit_rirs')
hf_hub_download('davidscripka/openwakeword_features', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
               repo_type='dataset', local_dir='/content')
hf_hub_download('davidscripka/openwakeword_features', 'validation_set_features.npy',
               repo_type='dataset', local_dir='/content')
print('listo')

## 5. Config YAML

In [ ]:
cfg = '''
target_phrase: ["oye matix"]
model_name: "oye_matix"
output_dir: "/content/my_custom_model"
piper_sample_generator_path: "./piper-sample-generator"
n_samples: 320
n_samples_val: 64
tts_batch_size: 50
rir_paths: ["/content/mit_rirs"]
background_paths: []
background_paths_duplication_rate: []
custom_negative_phrases: []
augmentation_rounds: 2
augmentation_batch_size: 16
feature_data_files:
  ACAV100M_sample: "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
false_positive_validation_data_path: "/content/validation_set_features.npy"
model_type: "dnn"
layer_size: 32
steps: 20000
max_negative_weight: 1500
target_accuracy: 0.6
target_recall: 0.25
target_false_positives_per_hour: 0.2
'''
open('/content/oye_matix.yaml', 'w').write(cfg)
print(cfg)

## 6. Aumentar + features

### Dummy generate_samples (para que train.py importe sin piper-sample-generator)

In [ ]:
# train.py importa 'generate_samples' (de piper-sample-generator) al inicio,
# aunque solo lo use con --generate_clips (que NO usamos: hicimos los audios
# con piper-tts). Creamos un modulo vacio para que el import no falle.
import os
d = '/content/openWakeWord/piper-sample-generator'
os.makedirs(d, exist_ok=True)
open(d + '/generate_samples.py', 'w').write(
    'def generate_samples(*a, **k):\n    raise RuntimeError("no se usa en modo augment")\n')
print('dummy generate_samples creado en', d)

### Descargar modelos base de openWakeWord (melspectrograma + embedding)

In [ ]:
# openWakeWord necesita sus modelos base (melspectrograma + embedding) para
# calcular features; con pip no vienen, hay que descargarlos.
import openwakeword.utils, glob
openwakeword.utils.download_models()
print('modelos base:', glob.glob('/content/openWakeWord/openwakeword/resources/models/*.onnx'))

In [ ]:
!cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --augment_clips

## 7. Entrenar el clasificador

In [ ]:
!cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --train_model

## 8. Verificar interfaz ONNX (entrada [1,16,96] f32, salida [1,1])

In [ ]:
import onnx, glob
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
m = onnx.load(p)
print('archivo:', p)
print('inputs :', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])

## 9. Guardar en Google Drive + descargar

Acepta el permiso para montar tu Drive. El archivo queda en **Mi unidad / oye_matix.onnx**. Desde ahi lo bajas a la PC sin riesgo de perderlo.

In [ ]:
import glob, shutil
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(p, '/content/drive/MyDrive/oye_matix.onnx')
    print('GUARDADO en tu Google Drive (Mi unidad) como oye_matix.onnx')
except Exception as e:
    print('No pude montar Drive:', e)
from google.colab import files
files.download(p)  # tambien intenta bajarlo directo al navegador

## 10. Integrarlo en Matix

Pasame `oye_matix.onnx` (ponlo en `MATIX/oye_matix.onnx` en la PC). Yo verifico, hago el swap en ambos pipelines, build y publico el APK nuevo.